# Assignment 2: Classical Control
Name: Samitha Ranasinghe

Purdue Username: sranasi


In this assignment, you will implement and tune a trajectory following controller for a 2-degree of freedom(DoF) robotic arm, and a race car.

## Getting Started

We'll be using OpenAI gym, along with PyBullet, to model the robot's environment.
- [OpenAI gym](https://gym.openai.com/) is a toolkit for developing planning and control algorithms. It provides a standard API that abstracts away the model of the robot's environment. It is primarily used for reinforcement learning agents, but can work with any controller, including the PD controllers you'll build in this assignment.
  - If you've never used gym, be sure to read this short [tutorial](https://gym.openai.com/docs/) before getting started.
- [PyBullet](https://pybullet.org/wordpress/) is an open-source physics engine that we'll use to model the robot.
- [pybullet-gym](https://github.com/benelot/pybullet-gym) is an open-source library that implements a variety of gym environments using PyBullet as the backend. One of these environments is `ReacherPyBulletEnv-v0`, the robotic arm that you'll be working with.

## Installation Instructions
Before running this notebook, please first create a conda environment with python version 3.8.
Then you'll need to install `gym` and `pybullet-gym`, like so:
```bash
~ # install gym
~ pip install gym==0.21.0
~ pip install numpy==1.21.1
~ # install pybullet-gym
~ git clone https://github.com/benelot/pybullet-gym.git
~ cd pybullet-gym
~ pip install -e .
~ pip install pybullet==3.2.1
```
We also included the environment.yml file that we used to run A2 for your reference.

## Overview: 2-DOF robotic arm

In [42]:
# load libraries (if this fails, see "Installation Instructions")
import gym
import numpy as np
import pybulletgym.envs
import matplotlib.pyplot as plt
import pybullet

In [43]:
# initialize the environment

# This try-except is to make sure there is only a single pybullet connection set-up
try:
    env.reset()
except NameError:
    env = gym.make("ReacherPyBulletEnv-v0")

obs = env.reset()
env.render(mode="human")
obs = env.reset()

In [44]:
pybullet.resetDebugVisualizerCamera(1, 5, -80, np.array([0,0,0]))

The robot arm you will be controlling looks like this:

![RobotArm](robotArm.png)

The base of the robot is at the origin; the links $l_0$ and $l_1$ are 0.1 and 0.11 units long respectively.

The action space of environment is [$\tau_0$,$\tau_1$], where $\tau_0$ and $\tau_1$ are the torques applied to joints $q_0$ and $q_1$ respectively.

Do _not_ use the observation space of the environment to get the robot's position. Instead, use the following class methods to obtain the joint angles:
```python
# To get the current position and angular velocity of q0
q0, q0_dot = env.unwrapped.robot.central_joint.current_position()
# To get the current position and angular velocity of q1
q1, q1_dot = env.unwrapped.robot.elbow_joint.current_position()

# To set joint q0 to a particular position. (Use only before running your controller, to initialize the start position)
env.unwrapped.robot.central_joint.reset_position(position, 0)
# To set joint q1 to a particular position. (Use only before running your controller, to initialize the start position)
env.unwrapped.robot.elbow_joint.reset_position(position, 0)
```

Your job is to implement PD controllers that track the trajectory

$$\begin{bmatrix}x(\theta) \\ y(\theta)\end{bmatrix}
= \begin{bmatrix}(0.19 + 0.02 \cos 4\theta)\cos\theta \\ (0.19 + 0.02 \cos 4\theta)\sin\theta\end{bmatrix},
\text{ for }\theta \in [-\pi, \pi]$$

This trajectory is plotted below:

In [45]:
x = [(0.19 + 0.02 * np.cos(theta * 4)) * np.cos(theta) for theta in np.arange(-np.pi, np.pi, 0.001)]
y = [(0.19 + 0.02 * np.cos(theta * 4)) * np.sin(theta) for theta in np.arange(-np.pi, np.pi, 0.001)]
plt.plot(x, y)
plt.axis('equal')
# plt.show()
plt.savefig('arm_traj.png')
traj = list(zip(x,y))

### 1. Forward Model

Derive the forward model for the robot as a closed-form expression expressed in joint angles and link length:

$$f\left(\begin{bmatrix}q_0 \\ q_1\end{bmatrix}\right) =
\begin{bmatrix}
% your answer here
0.1 \cos(q_0) + 0.11 \cos(q_0 + q_1) \\
0.1 \sin(q_0) + 0.11 \sin(q_0 + q_1)
\end{bmatrix}
= \begin{bmatrix} x \\ y\end{bmatrix}$$

Using the robot model parameters, write a function `getForwardModel` that takes the joint states and returns the end-effector position.

In [ ]:
def getForwardModel(q0, q1):
    l0 = 0.1
    l1 = 0.11

    x = l0 * np.cos(q0) + l1 * np.cos(q0 + q1)
    y = l0 * np.sin(q0) + l1 * np.sin(q0 + q1)

    return np.array([x, y])

### 2. Jacobian

Derive the expression for the Jacobian of the robot:

$$J_f(q_0, q_1) = \begin{bmatrix}
- 0.1 \sin(q_0) - 0.11 \sin(q_0 + q_1) & - 0.11 \sin(q_0 + q_1) \\
0.1 \cos(q_0) + 0.11 \cos(q_0 + q_1) & 0.11 \cos(q_0 + q_1)
\end{bmatrix}$$

Write a function `getJacobian` that takes the joint states and returns the Jacobian.

In [47]:
def getJacobian(q0, q1):
    l0 = 0.1
    l1 = 0.11

    j11 = - l0 * np.sin(q0) - l1 * np.sin(q0 + q1)
    j12 = - l1 * np.sin(q0 + q1)
    j21 = l0 * np.cos(q0) + l1 * np.cos(q0 + q1)
    j22 = l1 * np.cos(q0 + q1)

    jacobian = np.array([[j11, j12], [j21, j22]])

    return jacobian

### 3. X-Y controller

**Background:** for reasons beyond the scope of this course, it so happens that, for any robot,
$$\vec \tau = J^T \vec F,$$
where
- $\vec F = \langle F_x, F_y \rangle$ is the force vector exerted by the robot at the end effector
- $\vec \tau = \langle \tau_0, \tau_1 \rangle$ is the vector of torques exerted by the joints
- $J$ is the Jacobian matrix at the current position.

Use this fact to implement a closed-loop PD controller that controls the robot along the trajectory `traj`, using the error in the end-effector as the input signal. Your controller should compute forces $F_x$ and $F_y$, and then use `getJacobian` along with the above equation to translate them into joint torques.

Plot the trajectory of the robot juxtaposed over the desired trajectory, and calculate the mean square error between both paths. Also plot the errors with respect to time, and use those plots to tune your controller.

**Note: Initialize the robot arm to $q_0=\pi$ and $q_1=0$**

In [48]:
# Write your script here
import time

env.unwrapped.robot.central_joint.reset_position(np.pi, 0)
env.unwrapped.robot.elbow_joint.reset_position(0, 0)

kp = 7.0
kd = 0.03
dt = 1 / 240.0

last_err = np.zeros(2)

act_x = []
act_y = []
mse = []
t = []
i = 0

for targ_x, targ_y in traj:
    q0, _ = env.unwrapped.robot.central_joint.current_position()
    q1, _ = env.unwrapped.robot.elbow_joint.current_position()

    curr_pos = getForwardModel(q0, q1)
    targ_pos = np.array([targ_x, targ_y])

    err = targ_pos - curr_pos
    err_der = (err - last_err) / dt

    v = err * kp + err_der * kd

    q_dot = np.linalg.pinv(getJacobian(q0, q1)) @ v

    env.unwrapped.step(q_dot)

    act_x.append(curr_pos[0])
    act_y.append(curr_pos[1])

    mse.append(np.square(err).mean())
    t.append(i * dt)
    i += 1

    last_err = err

plt.figure()
plt.plot(act_x, act_y, label='Actual trajectory')
plt.plot(x, y, label='Given trajectory')
plt.axis('equal')
plt.legend()
plt.title('Trajectories')
plt.show()

plt.figure()
plt.plot(t, mse)
plt.title('MSE with time')
plt.show()

### 4. Inverse Kinematics

Using the functions `getForwardModel` and `getJacobian` from parts 1 and 2, write a function `getIK` that takes the current end-effector position, target end-effector position, and current joint states; and returns the target joint-states.

In [49]:
def getIK(current_position, target_position, current_state):
    
    error = target_position - current_position

    J = getJacobian(current_state[0], current_state[1])

    dq = np.linalg.pinv(J) @ error

    return current_state + dq

Now derive the analytical inverse kinematic solution; i.e. solve the problem using a closed-form equation, rather than an iterative method. _Show your work_. Correct answers without derivations will not receive full credit.

If we consider $(x,y)$ the end effector location and r the distance of the end effector from the origin,

$$ r^2 = x^2 + y^2 $$
because of law of cosines,
$$ r^2 = l_0^2 + l_1^2 - 2 l_0 l_1 \cos(180 - q_1) $$
and since $ cos(180 - q_1) = - cos(q_1) $,
$$ x^2 + y^2 = l_0^2 + l_1^2 + 2 l_0 l_1 \cos(q_1) $$
therefor,
$$ \cos(q_1) = \frac{x^2 + y^2 - l_0^2 - l_1^2}{2 l_0 l_1} $$
$$ q_1 = \pm \arccos(\frac{x^2 + y^2 - l_0^2 - l_1^2}{2 l_0 l_1}) $$

To find q0, we consider the angle of the end effector $(x,y)$ from the origin ($\alpha$) and then deduct the internal angle of the triangle ($\beta$),
$$ \tan(\alpha) = \frac{y}{x} $$
$$ \alpha = \arctan(\frac{y}{x}) $$
Then using law of sines,
$$ \frac{\sin(\beta)}{l_1} = \frac{\sin(180 - q_1)}{\sqrt(x^2 + y^2)} $$
which becomes,
$$ \sin(\beta) = \frac{l_1 \sin(q_1)}{\sqrt(x^2 + y^2)} $$
$$ \beta = \arcsin(\frac{l_1 \sin(q_1)}{\sqrt(x^2 + y^2)}) $$
and then,
$$ q_0 = \alpha - \beta $$
$$ q_0 = \arctan(\frac{y}{x}) - \arcsin(\frac{l_1 \sin(q_1)}{\sqrt(x^2 + y^2)}) $$

To conclude,

\begin{align*}
    % your derivations here
    q_1 &= \pm \arccos(\frac{x^2 + y^2 - l_0^2 - l_1^2}{2 l_0 l_1}) \\
    q_0 &= \arctan(\frac{y}{x}) - \arcsin(\frac{l_1 \sin(q_1)}{\sqrt(x^2 + y^2)})
\end{align*}

Explain what challenges there would be to use the analytical IK solution to track trajectories:

Because of the $\pm$ in the angle results, additional logic is needed to make sure the arm doesn't switch between elbow up and elbow down configurations for the same end effector position. 

### 5. IK controller

Implement a closed-loop PD controller that controls the robot along the trajectory `traj`, using the error in the joint-angles as the input signal.

Plot the trajectory of the robot juxtaposed over the actual trajectory and caluclate the mean square error between both paths. Also plot the errors with respect to time, and use those plots to tune your controller.

**Note: Initialize the robot arm to $q_0=\pi$ and $q_1=0$**

In [50]:
# Write your script here.
env.unwrapped.robot.central_joint.reset_position(np.pi, 0)
env.unwrapped.robot.elbow_joint.reset_position(0, 0)

kp = 7.0
kd = 0.03
dt = 1 / 240.0

last_err = np.zeros(2)

act_x = []
act_y = []
mse = []
t = []
i = 0

for targ_x, targ_y in traj:
    q0, _ = env.unwrapped.robot.central_joint.current_position()
    q1, _ = env.unwrapped.robot.elbow_joint.current_position()

    curr_q = np.array([q0, q1])
    curr_pos = getForwardModel(q0, q1)
    targ_pos = np.array([targ_x, targ_y])

    targ_q = getIK(curr_pos, targ_pos, curr_q)

    err = targ_q - curr_q
    err_der = (err - last_err) / dt

    v = err * kp + err_der * kd

    env.unwrapped.step(v)

    act_x.append(curr_pos[0])
    act_y.append(curr_pos[1])

    mse.append(np.square(err).mean())
    t.append(i * dt)
    i += 1

    last_err = err

plt.figure()
plt.plot(act_x, act_y, label='Actual trajectory')
plt.plot(x, y, label='Given trajectory')
plt.axis('equal')
plt.legend()
plt.title('Trajectories')
plt.show()

plt.figure()
plt.plot(t, mse)
plt.title('MSE with time')
plt.show()

### 6. Race Car

The objective of the `racecar` environment is to make the race car travel as far as possible on a track within the given time. There are 3 tracks vailable: `FigureEight`, `Linear`, and `Circle` (default).
Each track has a different shape, time limit, and horizon length. To set up an environment with a particular track, you can pass the track name while instantiating the environment. For example, to set up the figure eight trajectory:

```python
from racecar.SDRaceCar import SDRaceCar
env = SDRaceCar(render_env=True, track='FigureEight')
```

To install the race car environment, run the following commands:
```bash
~ git clone https://github.com/ucsdarclab/RaceCar.git
~ cd RaceCar
~ pip install -e .
```

The action space of the environment consists of [wheel angle, thrust]. The range of both these values are normalized to be between $\pm 1$.The observation space consists of [$x, y, \theta, v_x, v_y, \dot\theta, h$], where ($x, y, \theta$) is the intertial frame position of the car; $v_x, v_y$ are the longitudinal and lateral velocities  respectively; and $\dot\theta$ is the yaw rate. $h$ is the co-ordinate on the track the car has to reach.

At each time step, the race car environment gives a reward that is proportional to the speed of the car and its proximity to the track. It terminates (`done = True`) after a fixed amount of time, or if the car gets too far from the track.

Using these observations implement a controller that can traverse all three tracks. You may use different gains for different tracks, but the controller itself must be the same. Record the cumulative reward for each track; these rewards will be summed together to create your controller's score. If your controller has the highest score, you win!

_Tip:_ if you call `env.render()` at each step to visualize the car's path, you may find that Jupyter interprets each step as a separate image. To avoid this, try running
```IPython
%matplotlib tk # others include qt, wx, gtk, osx
```
to load results as animations in a separate window. You may have to experiment with several different backends to find the one that works best with your system.

In [51]:
from racecar.SDRaceCar import SDRaceCar

In [52]:
# define your controller here

def racecar_controller(obs, kp_steer, kd_steer, target_speed):
    x, y, theta, vx, vy, omega = obs[0:6]
    target_x, target_y = obs[6], obs[7]

    angle_to_target = np.arctan2(target_y - y, target_x - x)
    steer_error = (angle_to_target - theta + np.pi) % (2 * np.pi) - np.pi
    steer_action = (kp_steer * steer_error) - (kd_steer * omega)

    current_speed = np.sqrt(vx**2 + vy**2)
    
    dynamic_target_speed = target_speed * (1.0 - 0.6 * abs(steer_error))
    
    speed_error = dynamic_target_speed - current_speed
    
    thrust_action = 1.0 * speed_error 

    return np.array([np.clip(steer_action, -1.0, 1.0), 
                    np.clip(thrust_action, -1.0, 1.0)])

In [ ]:
%matplotlib tk
rc_env = SDRaceCar(render_env=True,track='Circle')
# run your controller for the 'Circle' environment

obs = rc_env.reset()
done = False
circ_reward = 0

while not done:
    action = racecar_controller(obs, 5, 0.5, 10.0)
    obs, reward, done, _ = rc_env.step(action)
    circ_reward += reward
    rc_env.render()

circ_reward

333.89130494344806

In [54]:
rc_env = SDRaceCar(render_env=True,track='Linear')
# run your controller for the 'Linear' environment
obs = rc_env.reset()
done = False
lin_reward = 0

while not done:
    action = racecar_controller(obs, 10, 0.01, 1000.0)
    obs, reward, done, _ = rc_env.step(action)
    lin_reward += reward
    rc_env.render()

lin_reward

73.87747072616382

In [55]:
rc_env = SDRaceCar(render_env=True,track='FigureEight')
# run your controller for the 'FigureEight' environment

obs = rc_env.reset()
done = False
eight_reward = 0

while not done:
    action = racecar_controller(obs, 10, 0.9, 9.0)
    obs, reward, done, _ = rc_env.step(action)
    eight_reward += reward
    rc_env.render()

eight_reward

353.75522447009035